# **Notebook 12 – Feature Leakage**

In [42]:
# Load Datset
import pandas as pd 
import numpy as np 
df = pd.read_csv("loan_application_approval.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Application_ID        3000 non-null   int64  
 1   Customer_ID           3000 non-null   int64  
 2   Age                   3000 non-null   int64  
 3   Annual_Income         2970 non-null   float64
 4   Credit_Score          2970 non-null   float64
 5   Employment_Years      2970 non-null   float64
 6   Loan_Amount           2970 non-null   float64
 7   Loan_Tenure_Months    3000 non-null   int64  
 8   Existing_Loans        3000 non-null   int64  
 9   Previous_Loans        3000 non-null   int64  
 10  Previous_Defaults     3000 non-null   int64  
 11  Previous_Loan_Amount  3000 non-null   int64  
 12  Application_Date      3000 non-null   object 
 13  Decision_Date         3000 non-null   object 
 14  Loan_Type             3000 non-null   object 
 15  Employment_Type      

## **1. What is Feature Leakage?**

**Understand the Concept**

- Feature leakage happens when a model uses future or unavailable information.
- It can give the model unrealistically high performance.

**Demonstrate the Concept**

- Approved_Amount before approval - Leakage
- Loan_Amount before approval - No Leakage

In [43]:
# Incorrect Feature Engineering - Leakage
df["Income_to_Approved_Amount"] = (df["Annual_Income"] / df["Approved_Amount"].replace(0, np.nan))
print(df[["Annual_Income","Approved_Amount","Income_to_Approved_Amount"]].head())

   Annual_Income  Approved_Amount  Income_to_Approved_Amount
0       254000.0                0                        NaN
1       407000.0                0                        NaN
2       498000.0           477000                   1.044025
3      1403000.0           557000                   2.518851
4       771000.0                0                        NaN


In [44]:
# Correct Feature Engineering - No Leakage
df["Income_to_Loan_Amount"] = (df["Annual_Income"] / df["Loan_Amount"].replace(0, np.nan))
print(df[["Annual_Income","Loan_Amount","Income_to_Loan_Amount"]].head())

   Annual_Income  Loan_Amount  Income_to_Loan_Amount
0       254000.0    1336000.0               0.190120
1       407000.0     249000.0               1.634538
2       498000.0     585000.0               0.851282
3      1403000.0     684000.0               2.051170
4       771000.0     706000.0               1.092068


**Explanation**

- Approved_Amount is known after approval, so it causes leakage.
- Loan_Amount is known at application time, so it is safe.
- Always use information available before prediction.

## **2. Target Leakage**

**Understand the Concept**

- Target leakage happens when a feature directly contains information about the target.
- This gives the model information it should not have.

**Demonstrate the Concept**

- Approved_Amount → Target Leakage
- Annual_Income → No Target Leakage

In [45]:
# Incorrect Feature Engineering → Leakage
df["Approval_Amount_Feature"] = df["Approved_Amount"]

# Correct Feature Engineering → No Leakage
df["Income_to_Loan_Ratio"] = (df["Annual_Income"] / df["Loan_Amount"].replace(0, np.nan))
print(df[["Approved_Amount","Approval_Amount_Feature","Annual_Income","Loan_Amount",    "Income_to_Loan_Ratio","Loan_Approved"]].head())

   Approved_Amount  Approval_Amount_Feature  Annual_Income  Loan_Amount  \
0                0                        0       254000.0    1336000.0   
1                0                        0       407000.0     249000.0   
2           477000                   477000       498000.0     585000.0   
3           557000                   557000      1403000.0     684000.0   
4                0                        0       771000.0     706000.0   

   Income_to_Loan_Ratio  Loan_Approved  
0              0.190120              0  
1              1.634538              0  
2              0.851282              1  
3              2.051170              1  
4              1.092068              0  


**Explanation**

- Approved_Amount is created after the approval decision, so it leaks the target.
- Income_to_Loan_Ratio uses information available before the decision.
- Use only pre-decision information when creating features.

## **3. Train-Test Leakage**

**Understand the Concept**

- Train-test leakage happens when test data influences the training process.
- This can make model results look better than they really are.

**Demonstrate the Concept**

- Scaling before splitting → Leakage
- Splitting before scaling → No Leakage

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["Age", "Annual_Income", "Credit_Score", "Loan_Amount"]].copy()
y = df["Loan_Approved"]
X = X.fillna(X.median())

In [47]:
# Incorrect Feature Engineering → Leakage
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [48]:
# Correct Feature Engineering → No Leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Explanation**

- In the incorrect method, the scaler learns from both train and test data.
- In the correct method, the scaler learns only from training data.

## **4. Temporal Leakage**

**Understand the Concept**

- Temporal leakage happens when future information is used to predict the past.
- Features should only use information available at that time.

**Demonstrate the Concept**

- Using Decision_Date to predict approval → Leakage
- Using Application_Date to create features → No Leakage

In [49]:
# Incorrect Feature Engineering → Leakage
df["Decision_Days"] = (pd.to_datetime(df["Decision_Date"]) -pd.to_datetime(df["Application_Date"])).dt.days

# Correct Feature Engineering → No Leakage
df["Application_Year"] = pd.to_datetime(df["Application_Date"]).dt.year
print(df[["Application_Date","Decision_Date","Decision_Days","Application_Year","Loan_Approved"]].head())

  Application_Date Decision_Date  Decision_Days  Application_Year  \
0       2024-11-06    2024-11-10              4              2024   
1       2025-11-11    2025-11-21             10              2025   
2       2024-01-04    2024-01-11              7              2024   
3       2025-09-20    2025-09-27              7              2025   
4       2023-03-23    2023-04-02             10              2023   

   Loan_Approved  
0              0  
1              0  
2              1  
3              1  
4              0  


**Explanation**

- Decision_Date is known after the loan decision, so it contains future information.
- Application_Year is available when the application is made.
- Future information should not be used for prediction.

## **5. Post-Outcome Features**

**Understand the Concept**

- Post-outcome features are created after the target outcome happens.
- Using them during prediction causes leakage.


In [50]:
# Incorrect Feature Engineering → Leakage
df["Approved_Amount_Ratio"] = (df["Approved_Amount"] / df["Loan_Amount"].replace(0, np.nan))

# Correct Feature Engineering → No Leakage
df["Income_Loan_Ratio"] = (df["Annual_Income"] / df["Loan_Amount"].replace(0, np.nan))
print(df[["Approved_Amount_Ratio","Income_Loan_Ratio","Loan_Approved"]].head())

   Approved_Amount_Ratio  Income_Loan_Ratio  Loan_Approved
0               0.000000           0.190120              0
1               0.000000           1.634538              0
2               0.815385           0.851282              1
3               0.814327           2.051170              1
4               0.000000           1.092068              0


**Explanation**

- Approved_Amount is available only after the decision.
- Annual_Income and Loan_Amount are available during the application.
- Post-outcome information should be excluded from prediction features.

## **6. Leakage Through Aggregations**

**Understand the Concept**

- Aggregation leakage happens when future records are included in a feature.
- Customer history should only use past information.

**Demonstrate the Concept**

- Total loans using all records → Leakage
- Previous loans using past records → No Leakage

In [51]:
# Sort by application date
df["Application_Date"] = pd.to_datetime(df["Application_Date"])
df = df.sort_values(["Customer_ID", "Application_Date"])

In [52]:
# Incorrect Feature Engineering → Leakage
df["Total_Customer_Loans"] = (df.groupby("Customer_ID")["Loan_Amount"].transform("sum"))

# Correct Feature Engineering → No Leakage
df["Previous_Loan_Count"] = (df.groupby("Customer_ID").cumcount())
print(df[["Customer_ID","Loan_Amount","Total_Customer_Loans","Previous_Loan_Count"]].head())

      Customer_ID  Loan_Amount  Total_Customer_Loans  Previous_Loan_Count
2274        50001    1604000.0             1604000.0                    0
2014        50003     580000.0             1719000.0                    0
67          50003     582000.0             1719000.0                    1
2018        50003     557000.0             1719000.0                    2
1972        50004     759000.0             1988000.0                    0


**Explanation**

- Total_Customer_Loans includes the current and future loan records.
- Previous_Loan_Count only represents earlier applications.
- Aggregations should use only past information.

## **7. Leakage Through Target Encoding**

**Understand the Concept**

- Target encoding replaces a category with its average target value.
- Calculating it using the full dataset can leak target information.

**Demonstrate the Concept**

- Encoding before train-test split → Leakage
- Encoding using training data only → No Leakage

In [53]:
# Incorrect Feature Engineering → Leakage
global_mean = df["Loan_Approved"].mean()
target_mean = df.groupby("Location")["Loan_Approved"].mean()
df["Location_Target_Encoded_Leakage"] = (df["Location"].map(target_mean))

In [54]:
# Correct Feature Engineering → No Leakage
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)

train_data = df.loc[train_idx]
test_data = df.loc[test_idx]

train_encoding = train_data.groupby("Location")["Loan_Approved"].mean()

df.loc[train_idx, "Location_Target_Encoded"] = (train_data["Location"].map(train_encoding).fillna(global_mean))
df.loc[test_idx, "Location_Target_Encoded"] = (test_data["Location"].map(train_encoding).fillna(global_mean))
print(df[["Location","Location_Target_Encoded_Leakage","Location_Target_Encoded"]].head())

        Location  Location_Target_Encoded_Leakage  Location_Target_Encoded
2274        Pune                         0.351899                 0.358025
2014     Chennai                         0.325203                 0.325342
67    Coimbatore                         0.268908                 0.259124
2018       Kochi                         0.310256                 0.324759
1972       Kochi                         0.310256                 0.324759


**Explanation**

- Full-data encoding uses target values from the test data.
- Training-only encoding avoids this leakage.
- Test data should not influence feature creation.

## **8. Detecting Leakage**

**Understand the Concept**

- Leakage can be found by checking when and how features are created.
- Very high model performance can also be a warning sign.

**Demonstrate the Concept**

- Check whether features use future information.
- Check whether features are created before or after splitting.

In [55]:
# Check features that may contain post-outcome information
possible_leakage = ["Approved_Amount","Final_Interest_Rate","Decision_Date"]
print("Possible leakage features:")
print(possible_leakage)

Possible leakage features:
['Approved_Amount', 'Final_Interest_Rate', 'Decision_Date']


In [56]:
# Check target relationship
print("\nCorrelation with target:")
print(df.select_dtypes(include="number").corr()["Loan_Approved"].sort_values(ascending=False))


Correlation with target:
Loan_Approved                      1.000000
Approved_Amount_Ratio              0.966477
Approved_Amount                    0.870471
Approval_Amount_Feature            0.870471
Credit_Score                       0.176345
Annual_Income                      0.156038
Income_Loan_Ratio                  0.131119
Income_to_Loan_Ratio               0.131119
Income_to_Loan_Amount              0.131119
Employment_Years                   0.128892
Age                                0.118620
Location_Target_Encoded_Leakage    0.069877
Location_Target_Encoded            0.068082
Decision_Days                      0.042883
Customer_ID                        0.015134
Previous_Loan_Count                0.015002
Application_Year                   0.014603
Application_ID                    -0.015692
Total_Customer_Loans              -0.016044
Loan_Tenure_Months                -0.017917
Previous_Loan_Amount              -0.032701
Existing_Loans                    -0.039478
Previo

**Explanation**

- Post-decision features should be checked carefully.
- Very strong relationships with the target can be a warning sign.
- Always check whether the feature was available at prediction time.

## **9. Preventing Leakage**

**Understand the Concept**

- Prevent leakage by using only information available before prediction.
- Split the data before learning information from the target.

**Demonstrate the Concept**

- Remove post-outcome features.
- Fit preprocessing only on training data.

In [57]:
# Remove features that are known after the decision
leakage_features = ["Approved_Amount","Final_Interest_Rate","Decision_Date"]
df_clean = df.drop(columns=leakage_features)

In [58]:
# Separate features and target
X = df_clean.drop(columns=["Loan_Approved"])
y = df_clean["Loan_Approved"]

In [59]:
# Split first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (2400, 28)
Testing data: (600, 28)


**Explanation**

- Post-outcome features are removed before model training.
- The data is split before further feature processing.
- This helps keep the training process free from leakage.